In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor, PATH_RESULTS_NON_RESIDUALS, PATH_RESULTS_RESIDUALS
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 15

eval_bs = 1000

stockGPT, stockGPT_params, opt1, sca1, sch1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3261440
5376
Continuing from previous checkpoint...


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 12:

Learning Rate: 4.00e-04



|██▌       | 25.0% (19:54) Evaluating model on validation data... (101/102) [1874/7496]:                  

Epoch 12:
Training Loss:
   (MAE) 0.002549779834225774
   (NLL) -3.7459828853607178
Validation Loss:
   (MAE) 0.002494317479431629
   (NLL) -3.7563579082489014

Best Validation: -4.029028415679932
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



...

|██████    | 60.0% (01:43) Evaluating model on validation data... (101/102) [5622/9370]: 

Epoch 13:
Training Loss:
   (MAE) 0.8659038543701172
   (NLL) 2078.122802734375
Validation Loss:
   (MAE) 0.8250589370727539
   (NLL) 16010.8505859375

Best Validation: 7.9027581214904785
----------------------------------------------------------------------------------------------------

Learning Rate: 5.00e-05



|████████  | 80.0% (02:12) Training LinearModel-B1... [7497/9370]:                       

Epoch 14:
Training Loss:
   (MAE) 0.8715372085571289
   (NLL) 6244.05224609375
Validation Loss:
   (MAE) 0.8313621282577515
   (NLL) 48017.55859375

Best Validation: 7.9027581214904785
----------------------------------------------------------------------------------------------------

Learning Rate: 2.50e-05



Epoch 15:
Training Loss:
   (MAE) 0.8748109936714172
   (NLL) 745.7900390625
Validation Loss:
   (MAE) 0.835011899471283
   (NLL) 5467.3759765625

Best Validation: 7.9027581214904785
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [5]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)


|██████████| 100.0% (04:31) Evaluating model on testing data... (51/52) [3120/3120]:                      

In [6]:
for key, features in [("NLL", StockGPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]),
                      ("MAE", StockGPT_cfg["target_features"]),
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockGPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockGPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-B1: 3.3M
    Training:       -4.4176  -4.7011  -4.7090  -4.6855    >  -4.6283
    Validation:     -4.4416  -4.7259  -4.7358  -4.7169    >  -4.6550
    Testing:        -4.4883  -4.7886  -4.7972  -4.7789    >  -4.7132
    
LinearModel-B1: 5.4K
    Training:       0.9503   14.0545  2.5143   13.0358    >  7.6387
    Validation:     0.9002   13.1510  2.2732   15.6799    >  8.0011
    Testing:        0.9017   13.3749  2.2215   15.4330    >  7.9828
    
NaiveModel-B1: 0
    Training:       2.5518   2.5522   2.5515   2.5518     >  2.5518
    Validation:     2.5915   2.5915   2.5914   2.5915     >  2.5915
    Testing:        2.6515   2.6514   2.6516   2.6515     >  2.6515
    

--------------------------

In [7]:
import importlib
import setup
importlib.reload(setup)
from setup import PATH_RESULTS_RESIDUALS

store_result(PATH_RESULTS_RESIDUALS, process_result(stockGPT, gpt_losses, gpt_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(linearModel, linear_losses, linear_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(naiveModel, naive_losses, naive_test_losses, max_epochs))

print(pd.read_parquet(PATH_RESULTS_RESIDUALS))

{'model': 'StockGPT-B1', 'bar_width': 1, 'train': {'NLL': [-4.417627811431885, -4.701135635375977, -4.708959102630615, -4.685479164123535], 'STD': [0.0049981107003986835, 0.003440731903538108, 0.0035511956084519625, 0.0035168800968676805], 'MAE': [0.001669945428147912, 0.0014917823718860745, 0.0014601312577724457, 0.0015618191100656986], 'PMAE': [46687140.0, 37121380.0, 35519500.0, 43891228.0]}, 'val': {'NLL': [-4.441615581512451, -4.7259063720703125, -4.735775947570801, -4.716869831085205], 'STD': [0.004884823691099882, 0.0033618626184761524, 0.003468469949439168, 0.0034339686390012503], 'MAE': [0.0016365264309570193, 0.0014437712961807847, 0.0013987576821818948, 0.0014967964962124825], 'PMAE': [57864652.0, 44776080.0, 42119804.0, 53234052.0]}, 'test': {'NLL': [-4.4882588386535645, -4.7886152267456055, -4.797151565551758, -4.778891086578369], 'STD': [0.004569426644593477, 0.0031316603999584913, 0.003198206890374422, 0.003189637092873454], 'MAE': [0.0013798344880342484, 0.0011651186505

In [8]:
from data_scrapper import scrape_data, get_all_tickers
from data_filler import fill_data
from data_preprocessor import preprocess_data
from setup import API_KEY, TIMEFRAME
import pandas_market_calendars as mcal

In [9]:
return ""
all_tickers = get_all_tickers("raw_data/all_tickers_trimmed_1_30", API_KEY)
scrape_data(API_KEY,
              "raw_data/data_5min_2026",
              250,
              all_tickers,
              mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index,
              TIMEFRAME)
fill_data("raw_data/data_5min_2026",
          "filled_raw_data/data_5min_2026",
          mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index)
preprocess_data("filled_raw_data/data_5min_2026",
                "preprocessed_data/data_5min_2026",
                mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index, [0.75, 0.9])

SyntaxError: 'return' outside function (521317128.py, line 1)

In [10]:
dls, train_norms = build_dataloaders("preprocessed_data/data_1min_2026")

Building DataLoaders...


In [11]:
test_losses = test_model(dls["test"], stockGPT, device, eval_bs)
print(test_losses)

({'NLL': tensor([-4.4540, -4.7459, -4.7551, -4.7289], device='cuda:0'), 'STD': tensor([0.0048, 0.0033, 0.0034, 0.0034], device='cuda:0'), 'MAE': tensor([0.0015, 0.0013, 0.0013, 0.0014], device='cuda:0'), 'PMAE': tensor([42139884., 31837770., 30846758., 39234888.], device='cuda:0')},)
